In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

import pandas as pd
import numpy as np

# Raw safety data for 10 Indian cities
data = {
    "City": [
        "Pune", "Mumbai", "Lucknow", "Kolkata",
        "Chennai", "Hyderabad", "Bhubaneswar",
        "Indore", "Jaipur", "Ahmedabad"
    ],

    "Population": [
        5050000, 18410000, 2901000, 14113000,
        8700000, 7750000, 856000,
        2170000, 3070000, 6350000
    ],

    "Total_Crimes": [
        13708, 51428, 14467, 11864,
        15765, 27031, 14416,
        12169, 27743, 22806
    ],

    "Violent_Crimes": [
        3573, 4750, 2236, 919,
        973, 1483, 844,
        1641, 2080, 2051
    ],

    "Road_Accidents": [
        1403, 2604, 1630, 1942,
        3762, 3785, 1291,
        3488, 3115, 1586
    ],

    "Road_Fatalities": [
        345, 370, 576, 191,
        542, 301, 350,
        290, 860, 410
    ]
}

df = pd.DataFrame(data)

display(df)


,City,Population,Total_Crimes,Violent_Crimes,Road_Accidents,Road_Fatalities
0,Pune,5050000,13708,3573,1403,345
1,Mumbai,18410000,51428,4750,2604,370
2,Lucknow,2901000,14467,2236,1630,576
3,Kolkata,14113000,11864,919,1942,191
4,Chennai,8700000,15765,973,3762,542
5,Hyderabad,7750000,27031,1483,3785,301
6,Bhubaneswar,856000,14416,844,1291,350
7,Indore,2170000,12169,1641,3488,290
8,Jaipur,3070000,27743,2080,3115,860
9,Ahmedabad,6350000,22806,2051,1586,410


In [2]:

# CELL 2: Calculate safety rates per 100,000 residents

df["Crime_Rate"] = (
    df["Total_Crimes"] / df["Population"] * 100000
)

df["Violent_Rate"] = (
    df["Violent_Crimes"] / df["Population"] * 100000
)

df["Accident_Rate"] = (
    df["Road_Accidents"] / df["Population"] * 100000
)

df["Fatality_Rate"] = (
    df["Road_Fatalities"] / df["Population"] * 100000
)

# Display calculated rates
display(df[[
    "City",
    "Crime_Rate",
    "Violent_Rate",
    "Accident_Rate",
    "Fatality_Rate"
]].round(2))


,City,Crime_Rate,Violent_Rate,Accident_Rate,Fatality_Rate
0,Pune,271.45,70.75,27.78,6.83
1,Mumbai,279.35,25.80,14.14,2.01
2,Lucknow,498.69,77.08,56.19,19.86
3,Kolkata,84.06,6.51,13.76,1.35
4,Chennai,181.21,11.18,43.24,6.23
5,Hyderabad,348.79,19.14,48.84,3.88
6,Bhubaneswar,1684.11,98.60,150.82,40.89
7,Indore,560.78,75.62,160.74,13.36
8,Jaipur,903.68,67.75,101.47,28.01
9,Ahmedabad,359.15,32.30,24.98,6.46


In [3]:

# CELL 3: Min-Max Normalization

def normalize(series):
    highest = series.max()
    lowest = series.min()

    if highest == lowest:
        return pd.Series(100.0, index=series.index)

    return 100 * (highest - series) / (highest - lowest)

# Crime indicators
df["N1"] = normalize(df["Crime_Rate"])
df["N2"] = normalize(df["Violent_Rate"])

# Road safety indicators
df["N3"] = normalize(df["Accident_Rate"])
df["N4"] = normalize(df["Fatality_Rate"])

# Display normalized scores
display(df[["City", "N1", "N2", "N3", "N4"]].round(2))


,City,N1,N2,N3,N4
0,Pune,88.29,30.24,90.46,86.14
1,Mumbai,87.80,79.05,99.74,98.34
2,Lucknow,74.09,23.37,71.13,53.20
3,Kolkata,100.00,100.00,100.00,100.00
4,Chennai,93.93,94.93,79.94,87.67
5,Hyderabad,83.46,86.29,76.13,93.60
6,Bhubaneswar,0.00,0.00,6.75,0.00
7,Indore,70.21,24.95,0.00,69.62
8,Jaipur,48.78,33.50,40.33,32.57
9,Ahmedabad,82.81,72.00,92.37,87.09


In [4]:

# CELL 4: Calculate Crime, Road Safety and Final Safety Indices

# Power Mean of Order 0.5
def root_mean(a, b):
    return ((np.sqrt(a) + np.sqrt(b)) / 2) ** 2

# Crime Index
df["CI"] = root_mean(df["N1"], df["N2"])

# Road Safety Index
df["RI"] = root_mean(df["N3"], df["N4"])

# Final Safety Index
df["SI"] = root_mean(df["CI"], df["RI"])

# Display results
display(df[["City", "CI", "RI", "SI"]].round(2))


,City,CI,RI,SI
0,Pune,55.47,88.29,70.93
1,Mumbai,83.37,99.04,91.03
2,Lucknow,45.17,61.84,53.18
3,Kolkata,100.00,100.00,100.00
4,Chennai,94.43,83.76,89.01
5,Hyderabad,84.87,84.64,84.75
6,Bhubaneswar,0.00,1.69,0.42
7,Indore,44.72,17.40,29.48
8,Jaipur,40.78,36.34,38.53
9,Ahmedabad,77.31,89.71,83.39


In [5]:

# CELL 5: Rank all cities by Safety Index

ranking = df[
    ["City", "CI", "RI", "SI"]
].copy()

ranking = ranking.sort_values(
    by="SI",
    ascending=False
).reset_index(drop=True)

ranking.insert(
    0, "Rank",
    range(1, len(ranking) + 1)
)

display(ranking.round(2))



,Rank,City,CI,RI,SI
0,1,Kolkata,100.00,100.00,100.00
1,2,Mumbai,83.37,99.04,91.03
2,3,Chennai,94.43,83.76,89.01
3,4,Hyderabad,84.87,84.64,84.75
4,5,Ahmedabad,77.31,89.71,83.39
5,6,Pune,55.47,88.29,70.93
6,7,Lucknow,45.17,61.84,53.18
7,8,Jaipur,40.78,36.34,38.53
8,9,Indore,44.72,17.40,29.48
9,10,Bhubaneswar,0.00,1.69,0.42


In [6]:

# CELL 6: Sensitivity test excluding Bhubaneswar

# Create a temporary dataset without Bhubaneswar
test = df[df["City"] != "Bhubaneswar"].copy()

# Recalculate normalization using the remaining 9 cities
test["N1"] = normalize(test["Crime_Rate"])
test["N2"] = normalize(test["Violent_Rate"])
test["N3"] = normalize(test["Accident_Rate"])
test["N4"] = normalize(test["Fatality_Rate"])

# Recalculate indices using the same square-root mean
test["CI"] = root_mean(test["N1"], test["N2"])
test["RI"] = root_mean(test["N3"], test["N4"])
test["SI"] = root_mean(test["CI"], test["RI"])

# Rank the 9 cities
test = test.sort_values(
    "SI", ascending=False
).reset_index(drop=True)

test["New_Rank"] = range(1, len(test) + 1)

# Compare old and new rankings
comparison = ranking[
    ranking["City"] != "Bhubaneswar"
][["City", "Rank", "SI"]].copy()

comparison = comparison.merge(
    test[["City", "New_Rank", "SI"]],
    on="City",
    suffixes=("_Original", "_Without_Bhubaneswar")
)

comparison["Rank_Change"] = (
    comparison["Rank"] - comparison["New_Rank"]
)

display(comparison.round(2))


,City,Rank,SI_Original,New_Rank,SI_Without_Bhubaneswar,Rank_Change
0,Kolkata,1,100.00,1,100.00,0
1,Mumbai,2,91.03,2,86.10,0
2,Chennai,3,89.01,3,85.71,0
3,Hyderabad,4,84.75,4,78.89,0
4,Ahmedabad,5,83.39,5,75.34,0
5,Pune,6,70.93,6,57.01,0
6,Lucknow,7,53.18,7,27.55,0
7,Jaipur,8,38.53,9,6.23,-1
8,Indore,9,29.48,8,14.66,1


In [7]:

# CELL 7: Leave-one-city-out sensitivity analysis

results = []

for removed_city in df["City"]:

    test = df[df["City"] != removed_city].copy()

    # Normalize again
    test["N1"] = normalize(test["Crime_Rate"])
    test["N2"] = normalize(test["Violent_Rate"])
    test["N3"] = normalize(test["Accident_Rate"])
    test["N4"] = normalize(test["Fatality_Rate"])

    # Calculate indices
    test["CI"] = root_mean(test["N1"], test["N2"])
    test["RI"] = root_mean(test["N3"], test["N4"])
    test["SI"] = root_mean(test["CI"], test["RI"])

    # Rank remaining cities
    test = test.sort_values("SI", ascending=False)
    test["New_Rank"] = range(1, len(test) + 1)

    # Compare original ranks, excluding removed city
    original = ranking[
        ranking["City"] != removed_city
    ][["City", "Rank"]].copy()

    # Adjust original ranks to a 1–9 scale
    removed_rank = int(
        ranking.loc[
            ranking["City"] == removed_city, "Rank"
        ].iloc[0]
    )

    original["Adjusted_Rank"] = (
        original["Rank"]
        - (original["Rank"] > removed_rank).astype(int)
    )

    comparison = original.merge(
        test[["City", "New_Rank"]],
        on="City"
    )

    changes = (
        comparison["Adjusted_Rank"]
        - comparison["New_Rank"]
    ).abs()

    results.append({
        "Removed_City": removed_city,
        "Cities_Changing_Rank": int((changes > 0).sum()),
        "Maximum_Rank_Change": int(changes.max()),
        "Average_Rank_Change": round(changes.mean(), 2)
    })

sensitivity = pd.DataFrame(results)

display(sensitivity.sort_values(
    "Average_Rank_Change",
    ascending=False
))


,Removed_City,Cities_Changing_Rank,Maximum_Rank_Change,Average_Rank_Change
6,Bhubaneswar,2,1,0.22
0,Pune,0,0,0.00
1,Mumbai,0,0,0.00
2,Lucknow,0,0,0.00
4,Chennai,0,0,0.00
3,Kolkata,0,0,0.00
5,Hyderabad,0,0,0.00
7,Indore,0,0,0.00
8,Jaipur,0,0,0.00
9,Ahmedabad,0,0,0.00


In [8]:

# CELL 8: Compare different aggregation methods

def power_mean(a, b, p):
    if p == 0:
        return np.sqrt(a * b)
    return ((a**p + b**p) / 2)**(1/p)

results = []

for p in [0, 0.5, 1]:
    ci = power_mean(df["N1"], df["N2"], p)
    ri = power_mean(df["N3"], df["N4"], p)
    si = power_mean(ci, ri, p)

    temp = pd.DataFrame({
        "City": df["City"],
        "SI": si
    })

    temp["Rank"] = temp["SI"].rank(
        ascending=False,
        method="min"
    ).astype(int)

    results.append(
        temp.set_index("City")["Rank"].rename(f"p={p}")
    )

comparison = pd.concat(results, axis=1)
comparison = comparison.sort_values("p=0.5")

display(comparison)


,p=0,p=0.5,p=1
City,,,
Kolkata,1,1,1
Mumbai,2,2,2
Chennai,3,3,3
Hyderabad,4,4,4
Ahmedabad,5,5,5
Pune,6,6,6
Lucknow,7,7,7
Jaipur,8,8,9
Indore,9,9,8


In [9]:

# CELL 9: Random Data Deletion Sensitivity Test

import numpy as np
import pandas as pd

np.random.seed(42)

indicators = [
    "Crime_Rate",
    "Violent_Rate",
    "Accident_Rate",
    "Fatality_Rate"
]

original_ranks = (
    df.set_index("City")["SI"]
    .rank(ascending=False, method="average")
)

results = []

for deletion_rate in [0.05, 0.10, 0.15]:

    rank_changes = []
    score_changes = []
    correlations = []

    for simulation in range(1000):

        test = df[["City"] + indicators].copy()

        # Randomly delete 5%, 10%, or 15% of values
        values = test[indicators].to_numpy().copy()
        total_values = values.size
        n_delete = round(total_values * deletion_rate)

        positions = np.random.choice(
            total_values, n_delete, replace=False
        )

        values.flat[positions] = np.nan
        test[indicators] = values

        # Replace missing values with column medians
        for col in indicators:
            test[col] = test[col].fillna(
                test[col].median()
            )

        # Recalculate normalization
        test["N1"] = normalize(test["Crime_Rate"])
        test["N2"] = normalize(test["Violent_Rate"])
        test["N3"] = normalize(test["Accident_Rate"])
        test["N4"] = normalize(test["Fatality_Rate"])

        # Recalculate Safety Index
        test["CI"] = root_mean(test["N1"], test["N2"])
        test["RI"] = root_mean(test["N3"], test["N4"])
        test["SI"] = root_mean(test["CI"], test["RI"])

        # Compare with original results
        test = test.set_index("City")
        new_ranks = test["SI"].rank(
            ascending=False, method="average"
        )

        rank_changes.append(
            (original_ranks - new_ranks).abs().mean()
        )

        score_changes.append(
            (df.set_index("City")["SI"] - test["SI"])
            .abs().mean()
        )

        correlations.append(
            original_ranks.corr(
                new_ranks, method="spearman"
            )
        )

    results.append({
        "Deletion": f"{int(deletion_rate * 100)}%",
        "Values_Deleted": n_delete,
        "Mean_Rank_Change": np.mean(rank_changes),
        "Mean_Score_Change": np.mean(score_changes),
        "Mean_Spearman_Rho": np.mean(correlations)
    })

deletion_results = pd.DataFrame(results)

display(deletion_results.round(3))


,Deletion,Values_Deleted,Mean_Rank_Change,Mean_Score_Change,Mean_Spearman_Rho
0,5%,2,0.185,2.001,0.982
1,10%,4,0.357,3.574,0.965
2,15%,6,0.521,5.198,0.947
